# Cough Classification with Simple CNN
## Quick 2-Day Implementation Guide

**Goal**: Classify cough recordings as Healthy, Mild, or Severe

**Dataset**: COUGHVID V3

**Architecture**: Lightweight CNN for mobile deployment

---

### Timeline:
- **Day 1**: Data preparation, preprocessing, model training (this notebook)
- **Day 2**: Fine-tuning, TFLite conversion, integration with backend

### Steps:
1. Setup and install dependencies
2. Load and explore COUGHVID dataset
3. Create custom labels (Healthy/Mild/Severe)
4. Extract Mel spectrograms
5. Build simple CNN model
6. Train and evaluate
7. Convert to TensorFlow Lite
8. Export for backend integration

## Step 1: Setup and Install Dependencies

In [ ]:
# Check if running on Colab
try:
    import google.colab
    IN_COLAB = True
    print("✅ Running on Google Colab")
except:
    IN_COLAB = False
    print("Running locally")

# Install required packages
!pip install -q librosa
!pip install -q pydub
!pip install -q kaggle

print("✅ Dependencies installed")

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import os
import json
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## Step 2: Download COUGHVID Dataset

**Instructions**:
1. Go to https://www.kaggle.com/datasets/orvile/coughvid-v3
2. Download the dataset manually OR use Kaggle API
3. Upload to Colab or use Kaggle API credentials

### Option A: Upload Kaggle API Key (Recommended for Colab)

In [ ]:
# Upload your kaggle.json file when prompted
if IN_COLAB:
    from google.colab import files
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()
    
    # Setup Kaggle credentials
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("✅ Kaggle credentials configured")

In [ ]:
# Download COUGHVID dataset
# Note: This is a large dataset (~2-3GB), download will take time
!kaggle datasets download -d orvile/coughvid-v3
!unzip -q coughvid-v3.zip -d coughvid_data
print("✅ Dataset downloaded and extracted")

### Option B: Manual Upload (If Kaggle API doesn't work)

In [ ]:
# If you manually downloaded the dataset, upload it here
# Uncomment and run if needed
# if IN_COLAB:
#     from google.colab import files
#     uploaded = files.upload()  # Upload the coughvid-v3.zip
#     !unzip -q coughvid-v3.zip -d coughvid_data

## Step 3: Explore and Prepare Dataset

In [ ]:
# Set data directory
DATA_DIR = 'coughvid_data'

# List files
print("Dataset contents:")
!ls -lh {DATA_DIR}

# Find metadata file
metadata_file = None
for file in os.listdir(DATA_DIR):
    if 'metadata' in file.lower() and file.endswith('.csv'):
        metadata_file = os.path.join(DATA_DIR, file)
        break

if metadata_file:
    print(f"\n✅ Found metadata file: {metadata_file}")
else:
    print("⚠️  Metadata file not found. Check dataset structure.")

In [ ]:
# Load metadata
metadata = pd.read_csv(metadata_file)

print(f"Total recordings: {len(metadata)}")
print(f"\nColumns: {metadata.columns.tolist()}")
print(f"\nFirst few rows:")
metadata.head()

In [ ]:
# Explore key columns
print("Status distribution:")
print(metadata['status'].value_counts())

print("\nCough detection statistics:")
print(metadata['cough_detected'].describe())

print("\nSNR (Signal-to-Noise Ratio) statistics:")
print(metadata['SNR'].describe())

## Step 4: Create Custom Labels (Healthy/Mild/Severe)

**Mapping Strategy**:
- **Healthy**: status='healthy'
- **Mild**: status='symptomatic' (no COVID)
- **Severe**: status='COVID' or symptomatic with respiratory_condition=True

In [ ]:
def create_severity_label(row):
    """Create custom severity labels"""
    status = row.get('status', '').lower()
    respiratory = row.get('respiratory_condition', False)
    
    if status == 'healthy':
        return 'Healthy'
    elif status == 'covid':
        return 'Severe'
    elif status == 'symptomatic':
        if respiratory:
            return 'Severe'
        else:
            return 'Mild'
    else:
        return 'Unknown'

# Apply labeling
metadata['severity'] = metadata.apply(create_severity_label, axis=1)

# Filter quality recordings
# Keep only recordings with:
# - High cough detection probability (> 0.8)
# - Good SNR (> 5)
# - Valid severity label
filtered_metadata = metadata[
    (metadata['cough_detected'] > 0.8) &
    (metadata['SNR'] > 5) &
    (metadata['severity'] != 'Unknown')
].copy()

print(f"Original recordings: {len(metadata)}")
print(f"After filtering: {len(filtered_metadata)}")
print(f"\nSeverity distribution:")
print(filtered_metadata['severity'].value_counts())

In [ ]:
# Balance dataset (optional - take equal samples from each class)
# This helps prevent bias towards majority class
min_samples = filtered_metadata['severity'].value_counts().min()
balanced_samples = min(min_samples, 1000)  # Cap at 1000 per class for speed

balanced_metadata = filtered_metadata.groupby('severity').sample(
    n=min(balanced_samples, filtered_metadata.groupby('severity').size().min()),
    random_state=42
)

print(f"Balanced dataset size: {len(balanced_metadata)}")
print(f"\nBalanced distribution:")
print(balanced_metadata['severity'].value_counts())

# Use balanced dataset for training
working_metadata = balanced_metadata.copy()

## Step 5: Audio Preprocessing and Feature Extraction

In [ ]:
# Configuration
TARGET_SR = 22050  # Sample rate
DURATION = 10  # seconds
N_MELS = 128  # Number of mel bands
N_FFT = 2048
HOP_LENGTH = 512

def load_and_preprocess_audio(file_path, target_sr=TARGET_SR, duration=DURATION):
    """Load and preprocess audio file"""
    try:
        # Load audio
        audio, sr = librosa.load(file_path, sr=target_sr, duration=duration)
        
        # Normalize
        audio = librosa.util.normalize(audio)
        
        # Trim silence
        audio, _ = librosa.effects.trim(audio, top_db=20)
        
        # Pad or truncate to fixed length
        target_length = target_sr * duration
        if len(audio) < target_length:
            audio = np.pad(audio, (0, target_length - len(audio)))
        else:
            audio = audio[:target_length]
        
        return audio, sr
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None, None

def extract_mel_spectrogram(audio, sr, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH):
    """Extract mel spectrogram from audio"""
    mel_spec = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=n_mels,
        n_fft=n_fft,
        hop_length=hop_length
    )
    
    # Convert to log scale (dB)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    # Normalize
    mel_spec_db = (mel_spec_db - mel_spec_db.mean()) / mel_spec_db.std()
    
    return mel_spec_db

print("✅ Preprocessing functions defined")

In [ ]:
# Test on one sample
# Find audio directory
audio_dir = None
for root, dirs, files in os.walk(DATA_DIR):
    if any(f.endswith(('.wav', '.webm', '.ogg')) for f in files):
        audio_dir = root
        break

if audio_dir:
    print(f"Audio directory: {audio_dir}")
    sample_files = [f for f in os.listdir(audio_dir) if f.endswith(('.wav', '.webm', '.ogg'))][:3]
    print(f"Sample files: {sample_files}")
    
    # Test preprocessing on first sample
    if sample_files:
        test_file = os.path.join(audio_dir, sample_files[0])
        audio, sr = load_and_preprocess_audio(test_file)
        if audio is not None:
            mel_spec = extract_mel_spectrogram(audio, sr)
            print(f"\n✅ Mel spectrogram shape: {mel_spec.shape}")
            
            # Visualize
            plt.figure(figsize=(10, 4))
            librosa.display.specshow(mel_spec, sr=sr, hop_length=HOP_LENGTH, x_axis='time', y_axis='mel')
            plt.colorbar(format='%+2.0f dB')
            plt.title('Mel Spectrogram')
            plt.tight_layout()
            plt.show()
else:
    print("⚠️  Audio directory not found")

## Step 6: Process All Audio Files and Create Dataset

In [ ]:
# Process all audio files
X = []  # Features (mel spectrograms)
y = []  # Labels (severity)
failed = 0

print("Processing audio files...")
for idx, row in tqdm(working_metadata.iterrows(), total=len(working_metadata)):
    # Get audio file path
    # COUGHVID uses UUID as filename
    uuid = row['uuid']
    
    # Try different extensions
    audio_file = None
    for ext in ['.wav', '.webm', '.ogg']:
        potential_path = os.path.join(audio_dir, f"{uuid}{ext}")
        if os.path.exists(potential_path):
            audio_file = potential_path
            break
    
    if audio_file is None:
        failed += 1
        continue
    
    # Load and process
    audio, sr = load_and_preprocess_audio(audio_file)
    if audio is None:
        failed += 1
        continue
    
    # Extract features
    mel_spec = extract_mel_spectrogram(audio, sr)
    
    X.append(mel_spec)
    y.append(row['severity'])

print(f"\n✅ Processed: {len(X)} files")
print(f"❌ Failed: {failed} files")

# Convert to numpy arrays
X = np.array(X)
y = np.array(y)

# Add channel dimension for CNN
X = X[..., np.newaxis]

print(f"\nFinal dataset shape: {X.shape}")
print(f"Labels shape: {y.shape}")

In [ ]:
# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_categorical = keras.utils.to_categorical(y_encoded)

print(f"Label mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"  {i}: {label}")

print(f"\nEncoded labels shape: {y_categorical.shape}")

In [ ]:
# Split dataset
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_categorical, test_size=0.3, random_state=42, stratify=y_encoded
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

## Step 7: Build Simple CNN Model

In [ ]:
def build_simple_cnn(input_shape, num_classes=3):
    """Build lightweight CNN for mobile deployment"""
    model = keras.Sequential([
        # Conv Block 1
        keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same', 
                           input_shape=input_shape),
        keras.layers.BatchNormalization(),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Dropout(0.25),
        
        # Conv Block 2
        keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        keras.layers.BatchNormalization(),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Dropout(0.25),
        
        # Conv Block 3
        keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        keras.layers.BatchNormalization(),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Dropout(0.25),
        
        # Dense layers
        keras.layers.Flatten(),
        keras.layers.Dense(256, activation='relu'),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

# Build model
model = build_simple_cnn(input_shape=X_train.shape[1:], num_classes=len(label_encoder.classes_))

# Compile
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
)

print("Model architecture:")
model.summary()

## Step 8: Train Model

In [ ]:
# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'best_cough_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print("Starting training...")
print("This may take 15-30 minutes depending on dataset size and GPU availability.\n")

In [ ]:
# Train model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,  # Will stop early if no improvement
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Training complete!")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Step 9: Evaluate Model

In [ ]:
# Evaluate on test set
test_loss, test_acc, test_precision, test_recall = model.evaluate(X_test, y_test, verbose=0)

print("Test Set Performance:")
print(f"  Accuracy: {test_acc:.4f}")
print(f"  Precision: {test_precision:.4f}")
print(f"  Recall: {test_recall:.4f}")
print(f"  Loss: {test_loss:.4f}")

In [ ]:
# Predictions
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

# Classification report
print("\nDetailed Classification Report:")
print(classification_report(
    y_test_classes, 
    y_pred_classes, 
    target_names=label_encoder.classes_
))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test_classes, y_pred_classes)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## Step 10: Convert to TensorFlow Lite

In [ ]:
# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Optimize for mobile
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

# Convert
tflite_model = converter.convert()

# Save
tflite_path = 'cough_classifier.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

model_size_mb = len(tflite_model) / (1024 * 1024)
print(f"✅ TFLite model saved: {tflite_path}")
print(f"Model size: {model_size_mb:.2f} MB")

In [ ]:
# Test TFLite model
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("TFLite Model Details:")
print(f"Input shape: {input_details[0]['shape']}")
print(f"Output shape: {output_details[0]['shape']}")

# Test on one sample
test_sample = X_test[0:1].astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], test_sample)
interpreter.invoke()
tflite_prediction = interpreter.get_tensor(output_details[0]['index'])

print(f"\nTest prediction:")
print(f"  Original model: {label_encoder.classes_[np.argmax(y_pred[0])]}")
print(f"  TFLite model: {label_encoder.classes_[np.argmax(tflite_prediction[0])]}")
print(f"  Ground truth: {label_encoder.classes_[y_test_classes[0]]}")
print("\n✅ TFLite model working correctly!")

## Step 11: Save Label Mapping and Model Info

In [ ]:
# Save label mapping
label_mapping = {i: label for i, label in enumerate(label_encoder.classes_)}

model_info = {
    'label_mapping': label_mapping,
    'input_shape': X_train.shape[1:],
    'target_sr': TARGET_SR,
    'duration': DURATION,
    'n_mels': N_MELS,
    'n_fft': N_FFT,
    'hop_length': HOP_LENGTH,
    'test_accuracy': float(test_acc),
    'test_precision': float(test_precision),
    'test_recall': float(test_recall),
    'model_size_mb': float(model_size_mb)
}

with open('model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print("✅ Model info saved to model_info.json")
print(json.dumps(model_info, indent=2))

## Step 12: Download Files for Backend Integration

In [ ]:
# Download files if on Colab
if IN_COLAB:
    from google.colab import files
    
    print("Downloading files...")
    files.download('cough_classifier.tflite')
    files.download('model_info.json')
    files.download('best_cough_model.h5')
    print("✅ Files downloaded!")
else:
    print("Files saved locally:")
    print("  - cough_classifier.tflite")
    print("  - model_info.json")
    print("  - best_cough_model.h5")

## Summary and Next Steps

### What We've Accomplished:
1. ✅ Loaded and explored COUGHVID dataset
2. ✅ Created custom labels (Healthy/Mild/Severe)
3. ✅ Extracted Mel spectrogram features
4. ✅ Built and trained simple CNN model
5. ✅ Evaluated model performance
6. ✅ Converted to TensorFlow Lite
7. ✅ Saved model files for deployment

### Files to Integrate with Backend:
- **cough_classifier.tflite** - TFLite model for inference
- **model_info.json** - Configuration and label mapping
- **best_cough_model.h5** - Full Keras model (optional backup)

### Next Steps (Day 2):
1. Copy `cough_classifier.tflite` and `model_info.json` to backend
2. Install Python dependencies on backend (`pip install tensorflow librosa`)
3. Integrate ML prediction into cough recording endpoint
4. Test with mobile app
5. Fine-tune if needed

### Expected Performance:
- Target Accuracy: 85-95%
- Model Size: <5 MB
- Inference Time: <1 second

---

**Great job! Your ML model is ready for deployment! 🚀**